# 00 · Shared data preparation

This notebook reads the Kaggle CSVs from `data/raw/` and saves prepared copies in `data/processed/`. The raw files stay unchanged. Notebook 01 uses the prepared training file for exploration; notebook 02 uses it for modeling.

## Load the data

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

train_raw = pd.read_csv(RAW / "train.csv")
test_raw = pd.read_csv(RAW / "test.csv")
print("Train:", train_raw.shape, "Test:", test_raw.shape)

Train: (42100, 19) Test: (28068, 18)


## Check the starting point

`booking_status` is the outcome in the training file: `1` means canceled and `0` means not canceled. The competition test file has no outcome.

In [2]:
assert "booking_status" in train_raw and "booking_status" not in test_raw
assert train_raw.drop(columns="booking_status").columns.equals(test_raw.columns)
assert train_raw.id.is_unique and test_raw.id.is_unique
assert train_raw.booking_status.isin([0, 1]).all()

print("Missing cells:", train_raw.isna().sum().sum())
print("Duplicate training rows:", train_raw.duplicated().sum())
print("Canceled:", train_raw.booking_status.sum(), "of", len(train_raw))

Missing cells: 0
Duplicate training rows: 0
Canceled: 16504 of 42100


## Add simple totals and flags

We keep unusual rows rather than guessing what a zero price or an impossible date means. The flags make those cases visible in later analysis.

In [3]:
def add_features(data):
    data = data.copy()
    data["total_guests"] = data.no_of_adults + data.no_of_children
    data["total_nights"] = data.no_of_weekend_nights + data.no_of_week_nights
    arrival = pd.to_datetime({"year": data.arrival_year, "month": data.arrival_month,
                              "day": data.arrival_date}, errors="coerce")
    data["invalid_arrival_date"] = arrival.isna().astype(int)
    data["zero_guests"] = data.total_guests.eq(0).astype(int)
    data["zero_nights"] = data.total_nights.eq(0).astype(int)
    data["zero_room_price"] = data.avg_price_per_room.eq(0).astype(int)
    return data

In [4]:
train = add_features(train_raw)
test = add_features(test_raw)
flags = ["invalid_arrival_date", "zero_guests", "zero_nights", "zero_room_price"]
print("Flagged training rows by condition:")
print(train[flags].sum())

Flagged training rows by condition:
invalid_arrival_date     50
zero_guests              16
zero_nights             114
zero_room_price         641
dtype: int64


## Save prepared copies

The generated CSVs are ignored by Git and can be recreated by rerunning this notebook. They preserve every original row and column.

In [5]:
assert train[train_raw.columns].equals(train_raw)
assert test[test_raw.columns].equals(test_raw)
PROCESSED.mkdir(parents=True, exist_ok=True)
train.to_csv(PROCESSED / "train_clean.csv", index=False)
test.to_csv(PROCESSED / "test_clean.csv", index=False)
print("Saved", PROCESSED / "train_clean.csv", train.shape)
print("Saved", PROCESSED / "test_clean.csv", test.shape)

Saved /Users/connerjamison/GitHub/CIS-8005-Project---Jalen-Conner/data/processed/train_clean.csv (42100, 25)
Saved /Users/connerjamison/GitHub/CIS-8005-Project---Jalen-Conner/data/processed/test_clean.csv (28068, 24)


## Data interface for modeling

Notebook 02 reads `data/processed/train_clean.csv`, assigns `booking_status` to `y`, and excludes `id` and `booking_status` from `X`. Meal plan, room type, and market segment are numeric category codes. The six added fields are `total_guests`, `total_nights`, `invalid_arrival_date`, `zero_guests`, `zero_nights`, and `zero_room_price`. Their usefulness for prediction has not yet been tested. The unlabeled `test_clean.csv` cannot provide evaluation metrics.